In [1]:
import kagglehub
import os
from pathlib import Path
import shutil

In [2]:
def download_kaggle_file(file_name: str):

    # Clear cache
    cache_dir = os.path.expanduser("~/.cache/kagglehub")

    if os.path.exists(cache_dir):
        shutil.rmtree(cache_dir)
        print("KaggleHub cache cleared.")
    else:
        print("No cache found.")

    # Download
    temp_path = kagglehub.dataset_download(file_name)
    
    # Set new directory
    data_dir = Path.cwd().parent / "data" / "source_data"
    
    # Move files
    print(f"Moving files from {temp_path} to {data_dir}...")
    
    files_moved = 0
    for filename in os.listdir(temp_path):
        source = os.path.join(temp_path, filename)
        destination = os.path.join(data_dir, filename)
        
        # Check if it's a file (skip subdirectories if any)
        if os.path.isfile(source):
            shutil.move(source, destination)
            print(f"Moved: {filename}")
            files_moved += 1
            
    print(f"\nFinished. {files_moved} files are now in source data folder")

In [3]:
download_kaggle_file("logandonaldson/sports-stadium-locations")

KaggleHub cache cleared.


100%|██████████| 3.20k/3.20k [00:00<00:00, 2.44MB/s]

Extracting files...
Moving files from /Users/lukebenson/.cache/kagglehub/datasets/logandonaldson/sports-stadium-locations/versions/1 to /Users/lukebenson/Desktop/UCL/Spatial_Data_Science/assessment/data/source_data...
Moved: stadiums.csv

Finished. 1 files are now in source data folder


In [4]:
import pandas as pd
import geopandas as gpd

In [5]:
pro_stadiums = pd.read_csv("../data/source_data/stadiums.csv")

In [6]:
pro_stadiums.head()

,Team,League,Division,Lat,Long
0,Dallas Mavericks,NBA,West,32.790556,-96.810278
1,Orlando Magic,NBA,East,28.539167,-81.383611
2,San Antonio Spurs,NBA,West,29.426944,-98.437500
3,Denver Nuggets,NBA,West,39.748920,-105.008400
4,Brooklyn Nets,NBA,East,40.682661,-73.975225


In [7]:
nba_stadiums = gpd.GeoDataFrame(
    pro_stadiums[pro_stadiums["League"] == "NBA"],
    geometry=gpd.points_from_xy(pro_stadiums[pro_stadiums["League"] == "NBA"]['Long'], pro_stadiums[pro_stadiums["League"] == "NBA"]['Lat']),
    crs="EPSG:4326")[["Team", "geometry"]]

In [8]:
nba_stadiums

,Team,geometry
0,Dallas Mavericks,POINT (-96.81028 32.79056)
1,Orlando Magic,POINT (-81.38361 28.53917)
2,San Antonio Spurs,POINT (-98.4375 29.42694)
3,Denver Nuggets,POINT (-105.0084 39.74892)
4,Brooklyn Nets,POINT (-73.97522 40.68266)
5,Washington Wizards,POINT (-77.0209 38.8982)
6,Golden State Warriors,POINT (-122.3875 37.76806)
7,Los Angeles Clippers,POINT (-118.26722 34.04306)
8,Los Angeles Lakers,POINT (-118.26722 34.04306)
9,Memphis Grizzlies,POINT (-90.05056 35.13833)


In [9]:
nba_stadiums.loc[nba_stadiums['Team'] == 'Los Angeles Clippers', 'Team'] = "LA Clippers"
nba_stadiums.loc[nba_stadiums['Team'] == 'Sacremento Kings', 'Team'] = "Sacramento Kings"

In [10]:
import time
from nba_api.stats.endpoints import leaguestandings

In [11]:
def get_nba_win_history(years=5):
    # Determine the current year for the 2025-26 season
    current_year = 2025
    all_seasons_data = []

    for i in range(years):
        # Format season string as 'YYYY-YY' (e.g., '2024-25')
        year_start = current_year - i
        year_end = str(year_start + 1)[2:]
        season_str = f"{year_start}-{year_end}"
        
        print(f"Fetching standings for {season_str}...")
        
        try:
            # Call the league standings endpoint
            standings = leaguestandings.LeagueStandings(season=season_str)
            df = standings.get_data_frames()[0]

            # Extract relevant columns
            season_df = df[['TeamCity', 'TeamName', 'WINS', 'LOSSES']].copy()
            season_df['Season'] = season_str
            season_df['Team'] = df['TeamCity'] + ' ' + df['TeamName']
            all_seasons_data.append(season_df)
            
            # Brief sleep to avoid hitting NBA.com rate limits
            time.sleep(1)
            
        except Exception as e:
            print(f"Could not retrieve data for {season_str}: {e}")

    # Combine all seasons into one DataFrame
    final_df = pd.concat(all_seasons_data, ignore_index=True)
    
    return final_df

In [12]:
win_pct_history = get_nba_win_history(5)

Fetching standings for 2025-26...
Fetching standings for 2024-25...
Fetching standings for 2023-24...
Fetching standings for 2022-23...
Fetching standings for 2021-22...


In [13]:
team_success = (
    win_pct_history.groupby("Team")
      .agg(
          total_wins=("WINS", "sum"),
          total_losses=("LOSSES", "sum")
      )
)

# Add win percentage
team_success["win_pct"] = team_success["total_wins"] / (team_success["total_wins"] + team_success["total_losses"])

team_success = team_success.reset_index(names='Team')

In [14]:
team_success

,Team,total_wins,total_losses,win_pct
0,Atlanta Hawks,206,204,0.502439
1,Boston Celtics,289,121,0.704878
2,Brooklyn Nets,167,243,0.407317
3,Charlotte Hornets,154,256,0.375610
4,Chicago Bulls,195,215,0.475610
5,Cleveland Cavaliers,259,151,0.631707
6,Dallas Mavericks,205,205,0.500000
7,Denver Nuggets,262,148,0.639024
8,Detroit Pistons,158,252,0.385366
9,Golden State Warriors,228,182,0.556098


In [26]:
from nba_api.stats.endpoints import franchisehistory

In [36]:
def get_nba_history():

    # fetch franchise history data
    history = franchisehistory.FranchiseHistory()
    df_history = history.get_data_frames()[0]

    # active teams
    df_2025 = df_history[df_history["END_YEAR"] == "2025"].copy()
    df_sorted = df_2025.sort_values(by="START_YEAR", ascending=True)

    # 3. Drop duplicates based on the city/name combination
    # 'keep=first' retains the row with the earliest START_YEAR
    unique_teams = df_sorted.drop_duplicates(subset=["TEAM_CITY", "TEAM_NAME"], keep="first").reset_index()

    # 4. Select the specific columns you need
    result = unique_teams[["TEAM_CITY", "TEAM_NAME", "YEARS", "LEAGUE_TITLES"]].rename(columns={
        'YEARS': 'age',
        'LEAGUE_TITLES': 'championships'
    })

    result["Team"] = result["TEAM_CITY"] + ' ' + result["TEAM_NAME"]

    return result[["Team", "age", "championships"]]

In [37]:
team_history = get_nba_history()

In [38]:
team_history

,Team,age,championships
0,New York Knicks,80,2
1,Golden State Warriors,80,7
2,Boston Celtics,80,18
3,Detroit Pistons,78,3
4,Los Angeles Lakers,78,17
5,Sacramento Kings,78,1
6,Atlanta Hawks,77,1
7,Philadelphia 76ers,77,3
8,Washington Wizards,65,1
9,Chicago Bulls,60,6


In [39]:
import requests
from io import StringIO

In [40]:
def get_nba_valuation(url: str):
    # Spoof a browser User-Agent to bypass the 403
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0 Safari/537.36"
        )
    }

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    # Parse tables from the response text
    tables = pd.read_html(StringIO(response.text))
    print(f"Found {len(tables)} table(s)")

    # Inspect and pick the right table
    for i, df in enumerate(tables):
        print(f"\n--- Table {i} ({df.shape[0]} rows × {df.shape[1]} cols) ---")
        print(df.head(3))

    # Use table 0 (adjust index if needed)
    df = tables[0]

    # Flatten multi-level headers if present
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [" ".join(str(c) for c in col).strip() for col in df.columns]

    return df

In [41]:
url = "https://en.wikipedia.org/wiki/Forbes_list_of_the_most_valuable_NBA_teams"
wiki_nba_valuation = get_nba_valuation(url)

Found 3 table(s)

--- Table 0 (30 rows × 7 cols) ---
   Rank                   Team State / Province          Value Change  \
0     1  Golden State Warriors       California    $11 billion    25%   
1     2     Los Angeles Lakers       California    $10 billion    41%   
2     3        New York Knicks         New York  $9.75 billion    30%   

        Revenue Operating Income  
0  $880 million     $409 million  
1  $551 million     $170 million  
2  $532 million      $98 million  

--- Table 1 (11 rows × 3 cols) ---
  vteForbes magazine                               vteForbes magazine.1  \
0          Companies                      Forbes Global 2000 Forbes 500   
1             People  The World's Billionaires Forbes 400 30 Under 3...   
2      Entertainment  General Forbes Top 40 Celebrity 100 Forbes Fic...   

   vteForbes magazine.2  
0                   NaN  
1                   NaN  
2                   NaN  

--- Table 2 (5 rows × 2 cols) ---
         0                            

In [42]:
wiki_nba_valuation

,Rank,Team,State / Province,Value,Change,Revenue,Operating Income
0,1,Golden State Warriors,California,$11 billion,25%,$880 million,$409 million
1,2,Los Angeles Lakers,California,$10 billion,41%,$551 million,$170 million
2,3,New York Knicks,New York,$9.75 billion,30%,$532 million,$98 million
3,4,Los Angeles Clippers,California,$7.5 billion,36%,$569 million,$154 million
4,5,Boston Celtics,Massachusetts,$6.7 billion,12%,$458 million,$116 million
5,6,Chicago Bulls,Illinois,$6.0 billion,20%,$434 million,$160 million
6,7,Houston Rockets,Texas,$5.9 billion,20%,$467 million,$191 million
7,8,Miami Heat,Florida,$5.7 billion,34%,$417 million,$110 million
8,9,Brooklyn Nets,New York,$5.6 billion,17%,$402 million,$50 million
9,10,Philadelphia 76ers,Pennsylvania,$5.45 billion,18%,$472 million,$204 million


In [43]:
team_valuation = wiki_nba_valuation[["Team", "Value"]]

In [44]:
def convert_currency_string(val):
    if pd.isna(val) or not isinstance(val, str):
        return val
    
    # Remove '$', 'billion', 'million', and any extra whitespace
    clean_val = val.replace('$', '').replace(' billion', '').strip()
    
    try:
        # Convert to float
        number = float(clean_val)
        
        # Scale based on the suffix
        if 'billion' in val.lower():
            return number * 1_000_000_000
        elif 'million' in val.lower():
            return number * 1_000_000
        return number
    except ValueError:
        return None

In [45]:
team_valuation['Value'] = team_valuation['Value'].apply(convert_currency_string)

In [46]:
team_valuation.loc[team_valuation['Team'] == 'Los Angeles Clippers', 'Team'] = "LA Clippers"

In [47]:
team_valuation.head()

,Team,Value
0,Golden State Warriors,1.100000e+10
1,Los Angeles Lakers,1.000000e+10
2,New York Knicks,9.750000e+09
3,LA Clippers,7.500000e+09
4,Boston Celtics,6.700000e+09


In [48]:
team_full = gpd.GeoDataFrame(team_valuation.merge(
    team_success[["Team", "win_pct"]], on="Team"
    ).merge(
        team_history, on="Team"
        ).merge(
            nba_stadiums, on="Team"
            )
)
team_full.columns = team_full.columns.str.lower()

In [49]:
team_full.head()

,team,value,win_pct,age,championships,geometry
0,Golden State Warriors,1.100000e+10,0.556098,80,7,POINT (-122.3875 37.76806)
1,Los Angeles Lakers,1.000000e+10,0.551220,78,17,POINT (-118.26722 34.04306)
2,New York Knicks,9.750000e+09,0.580488,80,2,POINT (-73.99361 40.75056)
3,LA Clippers,7.500000e+09,0.558537,56,0,POINT (-118.26722 34.04306)
4,Boston Celtics,6.700000e+09,0.704878,80,18,POINT (-71.06223 42.3663)


In [50]:
team_full.explore(
    column='value', 
    cmap='YlGnBu',
    tiles='CartoDB Positron',
    marker_kwds={
        'radius': 10,
        'fill': True
    },
    style_kwds={
        'stroke': True,
        'weight': 1,
        'color': 'white'
    }
)

In [51]:
team_full.to_csv('../data/processed_data/team_node.csv', index=False)